# SCORE-Net training — Synthetic (ablation: without-l1-feature-loss)

This notebook trains one model and writes all artifacts to its own folder:

* `global-best-weights*.pth` — model with the best **validation** accuracy across pruning.
* `final-iter-best-weights*.pth` — final pruning iteration (highest sparsity).
* `training_history*.json` — sparsity, validation accuracy, and rule count for each iteration.
* `*-pruned.pth` — compacted copies after compression.

The test set is evaluated only once, at the end; early stopping and model selection use `val_loader`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

# Locate the project root regardless of the notebook's working directory.
cwd = Path.cwd().resolve()
project_root = next(
    (path for path in [cwd, *cwd.parents] if (path / "utils" / "experiments.py").exists()),
    cwd.parent,
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.reproducibility import set_seed
from utils.experiments import load_splits
from utils.model import SCORENet
from utils.train import run_iterative_pruning_pipeline
from utils.custom_utils import prune_and_compact_model

# Seed all randomness sources (Python, NumPy, PyTorch, CUDA).
SEED = set_seed(42)

if torch.cuda.is_available():
    device = "cuda"
    print(f"CUDA available! GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
elif torch.backends.mps.is_available():
    device = "mps"
    print("MPS available! GPU")
else:
    device = "cpu"
    print("CUDA not found. Running on CPU.")


CUDA available! GPU: NVIDIA GeForce GTX 1650
GPU count: 1


In [2]:
# ==============================================================================
# EXPERIMENT CONFIGURATION
# ==============================================================================
BATCH_SIZE = 256
H_HIDDEN = 1024
DROPOUT = 0.1
VAL_SIZE = 0.1
EXPECTED_N_FEATURES = 30

# Artifacts always go to this notebook's folder, independent of cwd.
ARTIFACT_DIR = project_root / "models/ablations"
HISTORY_PATH = ARTIFACT_DIR / "training_history-without-l1-feature-loss.json"
GLOBAL_CHECKPOINT = ARTIFACT_DIR / "global-best-weights-without-l1-feature-loss.pth"
FINAL_CHECKPOINT = ARTIFACT_DIR / "final-iter-best-weights-without-l1-feature-loss.pth"
GLOBAL_PRUNED = ARTIFACT_DIR / "global-best-weights-without-l1-feature-loss-pruned.pth"
FINAL_PRUNED = ARTIFACT_DIR / "final-iter-best-weights-without-l1-feature-loss-pruned.pth"

print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"H (initial)     : {H_HIDDEN}")
print(f"Dropout         : {DROPOUT}")


Artifact directory: /home/ahmet/Projects/ScoreNetSingle/models/ablations
H (initial)     : 1024
Dropout         : 0.1


In [3]:
# ==============================================================================
# DATA SPLITS (TRAIN / VALIDATION / TEST)
# ==============================================================================
# load_splits: split first, stratify, and fit StandardScaler on train only.
# This keeps validation and test data out of preprocessing.
split = load_splits("synthetic", val_size=VAL_SIZE)

X_train_tensor, y_train_tensor = split.X_train, split.y_train
X_val_tensor, y_val_tensor = split.X_val, split.y_val
X_test_tensor, y_test_tensor = split.X_test, split.y_test

N_FEATURES = split.n_features
N_CLASSES = split.n_classes
feature_names, scaler = split.feature_names, split.scaler

if N_FEATURES != EXPECTED_N_FEATURES:
    raise ValueError(
        f"Expected feature count {EXPECTED_N_FEATURES}, data pipeline returned {N_FEATURES} "
        "returned. If preprocessing changed, update this constant and the N value in the paper."
    )

print(f"Train / validation / test: {X_train_tensor.shape[0]} / {X_val_tensor.shape[0]} / {X_test_tensor.shape[0]}")
print(f"Feature count (N)         : {N_FEATURES}")
print(f"Class count (C)           : {N_CLASSES}")
print(f"Class distribution (train): {torch.bincount(y_train_tensor).tolist()}")

# Seeded generator for reproducible batch order.
generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    split.loader("train").dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator
)
val_loader = split.loader("val", batch_size=BATCH_SIZE)
test_loader = split.loader("test", batch_size=BATCH_SIZE)


Train / validation / test: 14400 / 1600 / 4000
Feature count (N)         : 30
Class count (C)           : 20
Class distribution (train): [718, 721, 722, 722, 715, 722, 718, 719, 716, 717, 727, 719, 719, 722, 717, 717, 723, 722, 722, 722]


In [4]:
# Model
# NOTE: data loading and splitting consume the global RNG. Re-seed here so
# model initialization is independent of the data path and reproducible
# (see README, "Reproducing the results").
set_seed(SEED)
model = SCORENet(N_FEATURES, H_HIDDEN, N_CLASSES, dropout_rate=DROPOUT, device=device).to(device)
print(model)


SCORENet(
  (feature_linear): Linear(in_features=30, out_features=1024, bias=False)
  (output_linear): Linear(in_features=1024, out_features=20, bias=False)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [5]:
# ==============================================================================
# ITERATIVE PRUNING AND RETRAINING
# ==============================================================================
# Model selection and early stopping use only val_loader.
global_best_model, final_iter_model = run_iterative_pruning_pipeline(
    model, train_loader, val_loader, device,
    prune_amount_feature=0.1,
    num_prune_iters=20,
    l1_lambda_feature=0.0,
    history_path=str(HISTORY_PATH),
)



=============== INITIAL TRAINING (BASE MODEL) STARTING ===============


Epoch [001/250] | Train Acc: 0.0953 | Val Acc: 0.1581 | LR: 0.001000


Epoch [002/250] | Train Acc: 0.2995 | Val Acc: 0.2569 | LR: 0.001000


Epoch [003/250] | Train Acc: 0.4576 | Val Acc: 0.3962 | LR: 0.001000


Epoch [004/250] | Train Acc: 0.5551 | Val Acc: 0.4606 | LR: 0.001000


Epoch [005/250] | Train Acc: 0.6274 | Val Acc: 0.5363 | LR: 0.001000


... (1,215 outputs left out)

Epoch [025/250] | Train Acc: 0.7922 | Val Acc: 0.7281 | LR: 0.000500


Epoch [026/250] | Train Acc: 0.7889 | Val Acc: 0.7175 | LR: 0.000500


Epoch [027/250] | Train Acc: 0.7938 | Val Acc: 0.7275 | LR: 0.000500


Epoch [028/250] | Train Acc: 0.7978 | Val Acc: 0.7275 | LR: 0.000500
[!] Early stopping triggered (20 epochs without improvement). Best in cycle: 0.7300

========================= TRAINING COMPLETE =========================
1. Model: best overall (Global Best Val Acc: 0.9469)
2. Model: 20. best of the final iteration (Final Pruned Model)

[HISTORY] Training history saved: /home/ahmet/Projects/ScoreNetSingle/models/ablations/training_history-without-l1-feature-loss.json


In [6]:
# Raw (pre-compaction) checkpoints loaded by the analysis notebooks.
torch.save(global_best_model.state_dict(), GLOBAL_CHECKPOINT)
torch.save(final_iter_model.state_dict(), FINAL_CHECKPOINT)
print(f"Saved: {GLOBAL_CHECKPOINT}")
print(f"Saved: {FINAL_CHECKPOINT}")


Saved: /home/ahmet/Projects/ScoreNetSingle/models/ablations/global-best-weights-without-l1-feature-loss.pth
Saved: /home/ahmet/Projects/ScoreNetSingle/models/ablations/final-iter-best-weights-without-l1-feature-loss.pth


In [7]:
# ==============================================================================
# COMPACTION (STRUCTURAL + EMPIRICAL DEAD-RULE CLEANUP, EXACT BIAS SHIFTING)
# ==============================================================================
global_best_model = prune_and_compact_model(global_best_model, train_loader)
final_iter_model = prune_and_compact_model(final_iter_model, train_loader)

torch.save(global_best_model.state_dict(), GLOBAL_PRUNED)
torch.save(final_iter_model.state_dict(), FINAL_PRUNED)
print(f"Saved: {GLOBAL_PRUNED}")
print(f"Saved: {FINAL_PRUNED}")



MODEL COMPACTION AND DEAD-RULE CLEANUP REPORT
Rule capacity before compaction (H): 1024
-> Structurally empty rules (<= 0.0001) : 0
-> Empirically dead rules (never triggered): 0
Total removed rules: 0 (removal rate: %0.0)
Active surviving rules (new H): 1024
No dead or empty rules to remove; the model is already compact.



MODEL COMPACTION AND DEAD-RULE CLEANUP REPORT
Rule capacity before compaction (H): 1024
-> Structurally empty rules (<= 0.0001) : 8
-> Empirically dead rules (never triggered): 17
Total removed rules: 25 (removal rate: %2.4)
Active surviving rules (new H): 999
[OK] Pruning and compaction complete.
-> Net bias contribution transferred to classes:
[ 0.0001  0.0004 -0.0003 -0.0001 -0.0006  0.      0.0001 -0.0563  0.002
  0.     -0.001  -0.0009  0.      0.0149 -0.0156  0.001   0.0008  0.0126
 -0.0002 -0.0718]
-> New rule count (H): 999

Saved: /home/ahmet/Projects/ScoreNetSingle/models/ablations/global-best-weights-without-l1-feature-loss-pruned.pth
Saved: /home/ahmet/Projects/ScoreNetSingle/models/ablations/final-iter-best-weights-without-l1-feature-loss-pruned.pth


In [8]:
# ==============================================================================
# TEST EVALUATION (final only, once)
# ==============================================================================
global_best_model.eval()
final_iter_model.eval()

with torch.no_grad():
    for model_name, trained_model in [
        ("Final Iter (compacted)", final_iter_model),
        ("Global Best (compacted)", global_best_model),
    ]:
        preds = trained_model(X_test_tensor.to(device)).argmax(dim=1).cpu()
        acc = (preds == y_test_tensor).float().mean().item()
        print(f"{model_name:26s} | H={trained_model.H:5d} | Test Accuracy: {acc:.4f}")


Final Iter (compacted)     | H=  999 | Test Accuracy: 0.7013
Global Best (compacted)    | H= 1024 | Test Accuracy: 0.9310
